In [14]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect("../data/raw/database.sqlite")

# Tabellen anzeigen
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

print(tables)

# Datentabelle laden
df = pd.read_sql_query("""
SELECT *
FROM Tweets;
""", conn)

# conn.close()

     name
0  Tweets


In [15]:
# fehlende werte
df.isnull().sum()

tweet_id                        0
airline_sentiment               0
airline_sentiment_confidence    0
negativereason                  0
negativereason_confidence       0
airline                         0
airline_sentiment_gold          0
name                            0
negativereason_gold             0
retweet_count                   0
text                            0
tweet_coord                     0
tweet_created                   0
tweet_location                  0
user_timezone                   0
dtype: int64

In [16]:
# doppelte zeilen
df.duplicated().sum()

np.int64(0)

In [17]:
# Leerzeichen am Anfang und Ende entfernen
df["tweet_location"] = df["tweet_location"].str.strip()

# Einheitliche Groß-/Kleinschreibung
df["tweet_location"] = df["tweet_location"].str.title()

# Verschiedene Schreibweisen vereinheitlichen
location_mapping = {
    "New York, Ny": "New York",
    "New York City": "New York",
    "New York City, Ny": "New York",
    "New York, New York": "New York",
    "Queens, New York": "New York"
}

df["tweet_location"] = df["tweet_location"].replace(location_mapping)

In [18]:
df["tweet_location"].value_counts().head(10)

tweet_location
                     4687
New York              493
Boston, Ma            165
Washington, Dc        158
Usa                   131
Chicago               108
Nyc                   108
San Francisco, Ca      97
Los Angeles, Ca        97
San Francisco          91
Name: count, dtype: int64

In [21]:
import sqlite3

conn_processed = sqlite3.connect("../data/processed/cleaned_database.sqlite")

df.to_sql(
    "Tweets",
    conn_processed,
    if_exists="replace",
    index=False
)

conn_processed.close()